# QLoRA Fine-Tuning & High-Throughput vLLM API Serving
### 4-bit Quantized SFT on Qwen2.5-3B + Background vLLM OpenAI-Compatible API Server in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook provides a complete two-phase workflow designed for stability, speed, and modularity:
1. **Phase 1: Isolated QLoRA Fine-Tuning (Steps 1–4)**: Installs **only** core training dependencies at the start. Loads `Qwen/Qwen2.5-3B-Instruct` in 4-bit NormalFloat (NF4), trains LoRA adapters on `mlabonne/guanaco-llama2-1k` using TRL's `SFTTrainer`, and evaluates the adapter using pure PyTorch.
2. **Phase 2: Dedicated vLLM API Serving (Steps 5–8)**: Clears PyTorch training memory, separately installs `vllm` and API dependencies, launches the high-throughput vLLM OpenAI-compatible API server in the background with dynamic LoRA support (`qlora-adapter`), and queries the API using standard HTTP requests (streaming and non-streaming).


---
## 1. Hardware Inspection & Core Fine-Tuning Dependencies

To avoid dependency resolution conflicts and CUDA wheel mismatches, we install **only** the lightweight fine-tuning packages at the beginning. vLLM and API dependencies are installed separately in Step 5 after training completes.

> [!NOTE]
> **Fast Startup**:
> Fine-tuning dependencies install in ~30 seconds, allowing you to begin training immediately without installing vLLM upfront.


In [ ]:
!nvidia-smi


In [ ]:
# Install core fine-tuning dependencies only (vLLM installed separately in Step 5)
!pip install -q \
    "torch>=2.4.0" \
    "transformers>=4.45.0,<5.0.0" \
    "datasets>=3.0.0" \
    "trl>=0.11.0" \
    "peft>=0.13.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=1.0.0" \
    "python-dotenv"

print("✓ Core fine-tuning dependencies installed successfully.")


---
## 2. Authentication & Environment Configuration

`Qwen/Qwen2.5-3B-Instruct` is ungated and open (Apache 2.0). Providing a Hugging Face token is optional, but recommended to avoid download rate limits and enable saving/pushing your trained adapter to the Hub.

> [!TIP]
> Always strip whitespace and Windows CRLF carriage returns (`\r`) when loading tokens to prevent HTTP header validation failures (`requests.exceptions.InvalidHeader`).


In [ ]:
import os
import getpass

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    print("Optional: Enter your Hugging Face Token (press Enter to skip):")
    entered_token = getpass.getpass("HF Token: ")
    if entered_token.strip():
        hf_token = entered_token.strip()

if hf_token:
    # Proactive sanitization against whitespace and carriage returns
    os.environ["HF_TOKEN"] = hf_token.strip()
    print("✓ HF_TOKEN loaded and sanitized.")
else:
    print("✓ Running unauthenticated (valid for open Qwen2.5 weights).")


---
## 3. QLoRA Fine-Tuning Pipeline (`app.py`)

### What makes QLoRA efficient?
1. **4-bit NormalFloat (NF4)**: An information-theoretically optimal quantile quantization scheme for normally distributed model weights.
2. **Double Quantization (DQ)**: Quantizes the quantization constants themselves, saving ~0.37 bits per parameter (~300MB on a 3B model).
3. **Paged Optimizers (`paged_adamw_8bit`)**: Allocates page-locked memory and automatically offloads optimizer state spikes to CPU RAM during memory pressure, preventing sudden out-of-memory (OOM) errors.
4. **LoRA Adapters**: Freezes all 3 billion base parameters and injects low-rank trainable decomposition matrices (`r=16`, `lora_alpha=32`) into attention blocks (`q_proj`, `v_proj`, etc.). Only the ~60MB adapter is trained and saved.


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

# Configuration
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_ID = "mlabonne/guanaco-llama2-1k"
OUTPUT_DIR = "./qlora-adapter-output"

print(f"1. Loading tokenizer for: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit NormalFloat Quantization Config (bitsandbytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

print("2. Loading base model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

# Prepare model for k-bit training and disable KV cache for gradient checkpointing
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# PEFT LoRA Config targeting standard linear attention and MLP projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Load training dataset
print(f"3. Loading dataset: {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train")
print(f"✓ Dataset loaded: {len(dataset)} instruction samples.")


In [ ]:
# Configure Supervised Fine-Tuning (SFT) hyperparameters
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",       # Prevents OOM memory spikes
    gradient_checkpointing=True,    # Drastically cuts activation VRAM
    report_to="none",
    num_train_epochs=1,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Starting QLoRA fine-tuning...")
trainer.train()

print(f"Saving LoRA adapter weights to: {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✓ Training complete! LoRA adapter weights saved.")


---
## 4. In-Notebook Interactive Evaluation & Artifact Inspection

Before spinning up the vLLM API server, we validate generation directly with pure PyTorch:
- **`model.eval()`**: Freezes LoRA dropout layers (`0.05`) to avoid random output degradation.
- **`model.config.use_cache = True`**: Re-enables the KV-cache (which was disabled during training for gradient checkpointing).
- **`repetition_penalty = 1.15` & `no_repeat_ngram_size = 3`**: Prevents token loops.
- **`TextStreamer`**: Streams generation directly into the Colab cell output.


In [ ]:
from transformers import TextStreamer

# 1. Clean up trainer activation memory
if "trainer" in locals():
    del trainer
torch.cuda.empty_cache()

# 2. Put model in evaluation mode
model.eval()

# 3. Re-enable KV caching for fast autoregressive generation
model.config.use_cache = True

def generate_response(
    prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.15,
    stream: bool = True,
) -> str:
    """Generates a response from the fine-tuned model using streaming or non-streaming."""
    messages = [{"role": "user", "content": prompt}]
    formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_input, return_tensors="pt").to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None

    # Handle end of turn tokens for Qwen2.5
    eos_token_ids = [tokenizer.eos_token_id]
    extra_eos = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(extra_eos, int) and extra_eos not in eos_token_ids:
        eos_token_ids.append(extra_eos)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_token_ids,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=3,
            streamer=streamer,
        )

    if not stream:
        generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return ""

test_prompt = "What are the core advantages of QLoRA fine-tuning?"
print(f"Prompt: {test_prompt}\n")
print("=" * 60)
print("Generated Response (Streaming):")
print("=" * 60)
generate_response(test_prompt, max_new_tokens=256, stream=True)


In [ ]:
# Interactive Test: Try your own prompt with full length (512 tokens)!
user_prompt = "Explain quantum computing in simple terms for a high school student."
print(f"User Prompt: {user_prompt}\n")
print("=" * 60)
print("Generated Response (Live Streaming):")
print("=" * 60)
generate_response(user_prompt, max_new_tokens=512, stream=True)


In [ ]:
import os

print(f"Inspecting saved adapter files in '{OUTPUT_DIR}':\n")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  - {fname:<30} ({size_mb:.2f} MB)")


---
## 5. Memory Cleanup & Dedicated vLLM API Dependencies Setup

Now that fine-tuning and PyTorch validation are complete, we transition to production API serving with vLLM.

> [!IMPORTANT]
> **Why install vLLM separately using `uv`?**
> 1. **Zero Upfront Conflict**: By keeping fine-tuning dependencies isolated in Step 1, training started without massive C++ / CUDA wheel resolution or compatibility hurdles.
> 2. **VRAM Reset**: Before initializing the vLLM engine, we explicitly purge the PyTorch training model from GPU memory to free all VRAM for KV cache blocks.
> 3. **Instant Pre-Built Binary Installation (~45s)**: We use `uv pip install --system` to pull pre-compiled binary wheels for the active Python version, completely avoiding the silent 45-minute C++/CUDA source compilation triggered when older version pins lack wheels for modern runtimes.
> 4. **Triton Compatibility**: `setuptools` is included to ensure Triton JIT driver compilation succeeds.


In [ ]:
import gc
import torch

print("1. Releasing PyTorch training model and clearing CUDA memory...")
if "model" in locals():
    del model
if "trainer" in locals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

free_vram_mb = torch.cuda.mem_get_info()[0] / (1024 * 1024)
total_vram_mb = torch.cuda.mem_get_info()[1] / (1024 * 1024)
print(f"✓ GPU memory cleared: {free_vram_mb:.1f} MB / {total_vram_mb:.1f} MB VRAM available for vLLM.")


In [ ]:
# 1. Uninstall stale preinstalled torchaudio to prevent CUDA version mismatch
# Colab includes torchaudio built for CUDA 12, which conflicts with vLLM's CUDA 13 PyTorch
!pip uninstall -y torchaudio

# 2. Separately install vLLM and API serving dependencies using uv (~45 seconds)
!pip install -q uv
!uv pip install --system "vllm" "fastapi" "uvicorn[standard]" "requests" "openai>=1.0.0"

print("✓ vLLM and API dependencies successfully installed!")


---
## 6. Launching the vLLM OpenAI-Compatible API Server with LoRA Support

vLLM provides an enterprise-grade OpenAI-compatible API server (`vllm.entrypoints.openai.api_server`) that runs as a high-throughput background daemon.

### Production Optimizations Applied:
- **`--dtype auto`**: Runs in native high-precision 16-bit (~6.0 GB VRAM), eliminating on-the-fly dequantization latency and fitting effortlessly in Colab GPUs (T4: 15GB, L4: 24GB, A100: 80GB).
- **`--max-model-len 2048`**: Caps sequence length to reserve maximum GPU memory for concurrent requests and KV cache.
- **`--enforce-eager`**: Bypasses slow CUDA graph capture for instant (~30s) server startup.
- **`--enable-lora` & `--lora-modules qlora-adapter=./qlora-adapter-output`**: Dynamically registers our fine-tuned adapter under the model ID `qlora-adapter`.


In [ ]:
import subprocess
import time
import requests
import os

health_url = "http://localhost:8000/health"
log_file_path = "vllm_api_server.log"

# 1. Check if server is already active from a previous execution
is_already_running = False
try:
    check_res = requests.get(health_url, timeout=2)
    if check_res.status_code == 200:
        is_already_running = True
        print("✓ vLLM API server is ALREADY online and ready on http://localhost:8000!")
except Exception:
    pass

if not is_already_running:
    # Adapter path definition (with fallback in case runtime was restarted)
    OUTPUT_DIR = globals().get("OUTPUT_DIR", "./qlora-adapter-output")
    MODEL_ID = globals().get("MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")

    # Command to launch official OpenAI-compatible vLLM API server
    # Using 'vllm serve' (the modern standard command in vLLM v0.29+)
    server_cmd = [
        "vllm", "serve", MODEL_ID,
        "--dtype", "auto",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.85",
        "--enforce-eager",
        "--enable-lora",
        "--lora-modules", f"qlora-adapter={OUTPUT_DIR}",
        "--host", "0.0.0.0",
        "--port", "8000",
    ]

    print("Launching vLLM API server in background...")
    with open(log_file_path, "w") as log_file:
        server_process = subprocess.Popen(server_cmd, stdout=log_file, stderr=subprocess.STDOUT)

    print(f"Server PID: {server_process.pid}. Awaiting health check (cold start takes ~60–90s on first download)...")

    start_time = time.time()
    is_ready = False

    # Allow up to 240s for initial checkpoint download + KV-cache allocation
    while time.time() - start_time < 240:
        if server_process.poll() is not None:
            print("\n❌ Server process terminated unexpectedly. Log excerpt:")
            with open(log_file_path) as f:
                print(f.read()[-2000:])
            break
        try:
            r = requests.get(health_url, timeout=2)
            if r.status_code == 200:
                is_ready = True
                print(f"\n✓ vLLM API Server is healthy and running on http://localhost:8000! (Boot time: {int(time.time() - start_time)}s)")
                break
        except Exception:
            pass
        time.sleep(3)
        print(".", end="", flush=True)

    if not is_ready and server_process.poll() is None:
        print("\n⚠ Server startup timed out. Check 'vllm_api_server.log' for details:")
        with open(log_file_path) as f:
            print(f.read()[-2000:])


---
## 7. Interacting with the vLLM API via HTTP Requests

With the server running on `http://localhost:8000`, any standard HTTP client, OpenAI SDK, or frontend (such as `chat.py` or `app.py`) can interact with it.


In [ ]:
import requests
import json

# 1. Query registered models on the API
models_response = requests.get("http://localhost:8000/v1/models")
print("Registered Models & LoRA Adapters on vLLM API:")
print(json.dumps(models_response.json(), indent=2))


In [ ]:
# 2. Complete non-streaming chat completion with the fine-tuned adapter
payload = {
    "model": "qlora-adapter",
    "messages": [
        {"role": "user", "content": "What are the core advantages of QLoRA fine-tuning?"}
    ],
    "max_tokens": 256,
    "temperature": 0.7,
    "top_p": 0.9,
}

response = requests.post("http://localhost:8000/v1/chat/completions", json=payload)
data = response.json()

print("vLLM API Response (qlora-adapter):")
print("=" * 60)
print(data["choices"][0]["message"]["content"])
print("=" * 60)
print(f"Token Usage: {data.get('usage', {})}")


In [ ]:
import sys

# 3. Live real-time token streaming via Server-Sent Events (SSE)
prompt = "Explain quantum computing in simple terms for a high school student."
print(f"Streaming Prompt: {prompt}\n")
print("=" * 60)
print("Live Streamed API Output:")
print("=" * 60)

stream_payload = {
    "model": "qlora-adapter",
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 512,
    "temperature": 0.7,
    "stream": True,
}

with requests.post("http://localhost:8000/v1/chat/completions", json=stream_payload, stream=True) as r:
    for line in r.iter_lines():
        if not line:
            continue
        line_str = line.decode("utf-8")
        if line_str.startswith("data: "):
            chunk_data = line_str[6:].strip()
            if chunk_data == "[DONE]":
                break
            try:
                parsed = json.loads(chunk_data)
                delta = parsed["choices"][0].get("delta", {}).get("content", "")
                if delta:
                    sys.stdout.write(delta)
                    sys.stdout.flush()
            except json.JSONDecodeError:
                pass
print("\n" + "=" * 60)


In [ ]:
# 4. Side-by-Side Comparison: Base Model vs. Fine-Tuned Adapter via the API
comparison_prompt = "Tell me a short story about an AI discovering emotions."

for model_id in ["Qwen/Qwen2.5-3B-Instruct", "qlora-adapter"]:
    label = "1. BASE MODEL (No Adapter)" if model_id != "qlora-adapter" else "2. FINE-TUNED ADAPTER ('qlora-adapter')"
    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)
    res = requests.post("http://localhost:8000/v1/chat/completions", json={
        "model": model_id,
        "messages": [{"role": "user", "content": comparison_prompt}],
        "max_tokens": 100,
        "temperature": 0.7,
    }).json()
    print(res["choices"][0]["message"]["content"].strip())


In [ ]:
# 5. Server Management Helper: Stop the background vLLM API server when finished
def stop_vllm_server():
    global server_process
    if "server_process" in globals() and server_process.poll() is None:
        print(f"Stopping vLLM API server (PID: {server_process.pid})...")
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print("✓ Server stopped successfully.")
        except subprocess.TimeoutExpired:
            server_process.kill()
            print("✓ Server process killed.")
    else:
        print("Server is not currently running.")

# Uncomment to stop server when finished:
# stop_vllm_server()


---
## 8. Exposing the vLLM API to External Clients (or your Laptop)

Because the vLLM API server conforms to the standard OpenAI `/v1` specification, you can expose it outside Google Colab using a secure tunnel. This lets you connect your local laptop terminal, frontend, or python scripts directly to your Colab GPU instance.

### Option A: Using `localtunnel` (Instant, No Account Required)

> [!IMPORTANT]
> **Key localtunnel requirements**:
> 1. **`-y` Flag**: Always run `npx` with `-y` (`!npx -y localtunnel --port 8000`) to auto-confirm package installation without hanging on interactive prompts.
> 2. **Bypass Header**: When querying the tunnel via `curl` or automated HTTP scripts, you **must** include the header `-H "Bypass-Tunnel-Reminder: true"`. Without it, localtunnel intercepts the request and returns an HTML reminder page instead of proxying to the vLLM API server.
> 3. **Browser Password**: If visiting the URL in a browser, localtunnel requires the host's public IP address as a one-time password (retrieved via `curl https://loca.lt/mytunnelpassword`).


In [ ]:
# 1. Print Colab public IP (acts as the password if opening the URL in a web browser)
!curl -s https://loca.lt/mytunnelpassword && echo ""

# 2. Expose the vLLM API server (port 8000) using localtunnel
# The '-y' flag automatically confirms package installation
!npx -y localtunnel --port 8000


### Calling the External URL from Your Local Laptop via `curl`

Once localtunnel displays your public URL (e.g., `https://funny-tiger-42.loca.lt`), you can call your fine-tuned model directly from your local terminal:

#### 1. Check Registered Models & LoRA Adapters
```bash
curl -s https://<your-subdomain>.loca.lt/v1/models \
  -H "Bypass-Tunnel-Reminder: true"
```

#### 2. Standard Chat Completion (Non-Streaming JSON)
```bash
curl -X POST https://<your-subdomain>.loca.lt/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Bypass-Tunnel-Reminder: true" \
  -d '{
    "model": "qlora-adapter",
    "messages": [
      {"role": "user", "content": "Explain quantum computing in simple terms for a high school student."}
    ],
    "max_tokens": 200,
    "temperature": 0.7
  }'
```

#### 3. Real-Time Token Streaming (`curl -N`)
Use `-N` (or `--no-buffer`) so `curl` prints tokens to your terminal in real time as the Colab GPU generates them:
```bash
curl -N -X POST https://<your-subdomain>.loca.lt/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Bypass-Tunnel-Reminder: true" \
  -d '{
    "model": "qlora-adapter",
    "messages": [
      {"role": "user", "content": "Write a clean Python function to calculate Fibonacci numbers."}
    ],
    "max_tokens": 256,
    "temperature": 0.7,
    "stream": true
  }'
```

---

### Option B: Using `pyngrok` (Requires Free ngrok Account)
```python
!pip install -q pyngrok
from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
public_url = ngrok.connect(8000).public_url
print(f"Public API URL: {public_url}/v1")
```

Once connected, you can also query this Colab vLLM server directly from Python using the official `openai` SDK:
```python
import openai

client = openai.OpenAI(
    base_url="https://<your-subdomain>.loca.lt/v1",
    api_key="none",
    default_headers={"Bypass-Tunnel-Reminder": "true"},
)
response = client.chat.completions.create(
    model="qlora-adapter",
    messages=[{"role": "user", "content": "Explain quantum computing in simple terms."}],
)
print(response.choices[0].message.content)
```


---
## 9. Troubleshooting & Production Engineering Reference

Below is the compilation of critical real-world issues diagnosed and resolved during pipeline development:

### 1. `ImportError: libcudart.so.13: cannot open shared object file: No such file or directory`
- **Root Cause**: An unconstrained `vllm>=0.6.0` on CUDA 12 installs CUDA 13.0 wheels.
- **Solution**: Install prebuilt wheels via `uv pip install --system vllm`.

### 2. `AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended`
- **Root Cause**: Transformers v5 removed `all_special_tokens_extended`.
- **Solution**: Pin `transformers>=4.45.0,<5.0.0`.

### 3. `ModuleNotFoundError: No module named 'setuptools'`
- **Root Cause**: Triton dynamically imports `setuptools` during backend driver discovery.
- **Solution**: Explicitly include `setuptools` in dependencies.

### 4. `requests.exceptions.InvalidHeader: Invalid leading whitespace, reserved character(s)`
- **Root Cause**: Windows CRLF line endings (`\r\n`) in `.env` attach `\r` to `$HF_TOKEN`.
- **Solution**: Call `.strip()` on `os.environ["HF_TOKEN"]` before use.

### 5. `ValueError: No available memory for the cache blocks` (KV Cache OOM)
- **Root Cause**: PyTorch training model not cleared before starting vLLM.
- **Solution**: Run `del model; del trainer; gc.collect(); torch.cuda.empty_cache()` in Step 5 before starting vLLM.

### 6. `RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions`
- **Root Cause**: Colab has preinstalled `torchaudio` built for CUDA 12.8, while vLLM brings PyTorch with CUDA 13.0.
- **Solution**: Run `!pip uninstall -y torchaudio` in Step 5.

### 7. `Value error, Unknown quantization method: bitsandbytes`
- **Root Cause**: Modern vLLM deprecated legacy `bitsandbytes` quantization in ModelConfig.
- **Solution**: Use `--dtype auto` (3B model runs in native 16-bit consuming only ~6 GB VRAM on Colab's 15GB+ GPUs).

### 8. `localtunnel` Hanging on Stdin Prompt
- **Root Cause**: Running `npx localtunnel` prompts interactively to confirm installation.
- **Solution**: Add `-y` (`!npx -y localtunnel --port 8000`). When querying via `curl`, pass `-H "Bypass-Tunnel-Reminder: true"`.
